In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from avito_retrieval.data.loader import (
    load_train, load_benchmark_queries, load_benchmark_items,
    build_item_text, build_query_text,
)

train = load_train()
bench_queries = load_benchmark_queries()
bench_items = load_benchmark_items()

print("train:", train.shape)
print("bench_queries:", bench_queries.shape)
print("bench_items:", bench_items.shape)

train.head(3)

train: (497673, 19)
bench_queries: (2452, 6)
bench_items: (189212, 14)


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,Вид услуги Место оказания услуг Первомайский п...,e8b685dffe1a408e,Скупаю практически любую современную новую и б...,114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,Вид услуги Место оказания услуг Нижний Новгоро...,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания ус...",624846856ce81d69,В ритме современной жизни так сложно найти мо...,114


In [7]:
cols_to_check = [
    "search_infm_params_text", "item_description_raw",
    "item_infm_params_text", "item_rating", "item_price",
]
for col in cols_to_check:
    if col in train.columns:
        empty_share = train[col].isna().mean()
        print(f"{col}: {empty_share:.2%} пустых")

search_infm_params_text: 0.00% пустых
item_description_raw: 0.00% пустых
item_infm_params_text: 0.00% пустых
item_rating: 5.67% пустых
item_price: 0.00% пустых


In [8]:
from transformers import AutoTokenizer
from avito_retrieval import config

tokenizer = AutoTokenizer.from_pretrained(config.BIENCODER_MODEL_NAME)


def token_lengths(texts: pd.Series, batch_size: int = 512) -> np.ndarray:
    """Считает длину в токенах батчами, без padding (реальная длина текста)."""
    lengths = []
    texts = texts.tolist()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, truncation=False, padding=False)
        lengths.extend(len(ids) for ids in enc["input_ids"])
    return np.array(lengths)


query_texts = build_query_text(train)
item_texts = build_item_text(train)

query_lens = token_lengths(query_texts)
item_lens = token_lengths(item_texts)


def print_percentiles(name: str, arr: np.ndarray):
    percentiles = [50, 75, 90, 95, 99, 100]
    values = np.percentile(arr, percentiles)
    print(f"\n{name}:")
    for p, v in zip(percentiles, values):
        print(f"  p{p}: {v:.0f} токенов")


print_percentiles("Длина запроса (query + params)", query_lens)
print_percentiles("Длина объявления (title + params + description)", item_lens)

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(query_lens, bins=50)
axes[0].set_title("Длина запроса, токены")
axes[0].axvline(np.percentile(query_lens, 95), color="red", linestyle="--", label="p95")
axes[0].legend()

# Обрезаем хвост для читаемости графика объявлений
axes[1].hist(np.clip(item_lens, 0, 300), bins=50)
axes[1].set_title("Длина объявления, токены (обрезано до 300)")
axes[1].axvline(np.percentile(item_lens, 95), color="red", linestyle="--", label="p95")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Если у одного текста запроса почти всегда одна категория объявления —
# это сигнал в пользу фильтрации кандидатов по категории до текстового поиска
query_to_category = (
    train.groupby("search_query")["item_category_id"]
    .agg(lambda x: x.value_counts(normalize=True).iloc[0])
)
print("Доля случаев, когда топ-категория покрывает >=80% выборов по запросу:")
print((query_to_category >= 0.8).mean())


# Средняя энтропия категорий на запрос (ниже = запрос сильнее определяет категорию)
def normalized_entropy(series):
    counts = series.value_counts(normalize=True)
    return -(counts * np.log(counts + 1e-12)).sum()


entropy_per_query = train.groupby("search_query")["item_category_id"].apply(normalized_entropy)
print("\nМедианная энтропия категории на запрос:", entropy_per_query.median())

In [ ]:
title_counts = bench_items["item_title_raw"].value_counts()
print("Топ-10 самых частых заголовков объявлений:")
print(title_counts.head(10))
print(f"\nДоля объявлений с неуникальным title: {(title_counts[title_counts > 1].sum() / len(bench_items)):.2%}")

In [ ]:
bench_items["item_category_id"].unique()

In [ ]:
bench_queries["search_category"].unique()

In [ ]:
bench_queries[bench_queries["search_category"] == 114]

In [9]:
bench_items

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,57.0,5.000000,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания ...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\...",114
1,Обучение ребенка чтению,7.0,5.000000,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услу...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ,...",114
2,Афрокудри 5+,14.0,5.000000,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания ус...",3f6ae81704565b9c,ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,114
3,Монтаж малых архитектурных форм,2.0,5.000000,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,False,False,Вид услуги Строительство Место оказания услуг ...,0618dec37a59a21a,Оказываем все виды услуг по сборке и монтажу и...,114
4,Репетитор по истории 10 класс ЕГЭ,19.0,5.000000,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,False,False,"Вид услуги Обучение, курсы Место оказания услу...",13da2c81574677ed,🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев!\n\n...,114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189207,3Д визуализация,3.0,5.000000,2000.000000000000000,2058739,53.143138890000003,659300,56.830345149999999,True,False,Вид услуги Строительство Место оказания услуг ...,ffbf3d333196e26e,"Меня зовут Азалия, я архитектор с высшим образ...",114
189208,Обучение наращиванию ресниц,26.0,4.346154,22000.000000000000000,2096704,41.946958000000002,655950,44.627530000000000,False,False,"Вид услуги Обучение, курсы Место оказания услу...",2be43871d237fcfe,Обучение наращиванию ресниц с нуля и курсом по...,114
189209,Репетитор начальные классы,46.0,4.956522,600.000000000000000,86456,45.511155290708302,632650,58.382799456130002,False,False,"Вид услуги Обучение, курсы Место оказания услу...",09cea44e7f461034,"Преподаватель высшей категории, опыт работы 40...",114
189210,Баянист на ваш праздник,6.0,5.000000,2000.000000000000000,2301563,48.405718999999998,653120,53.128261999999999,False,False,"Вид услуги Праздники, мероприятия Место оказан...",425f44923ac75293,Пoющий бaянист лaуpeaт кoнкурса Игрaй гаpмонь ...,114


In [12]:
bench_items['item_infm_params_text'].str.strip().eq("").mean()

np.float64(0.0)